# 📊 Notebook 1: YOLOv8-seg vs YOLOv11-seg Baselines Comparison
This notebook is 100% self-contained for Kaggle execution. It establishes clean baseline fine-tuning results (without KD) comparing **YOLOv8n-seg** vs **YOLOv11n-seg** on **Crack500** and **DeepCrack** datasets separately.

### Experiments in this Notebook:
1. `v8_crack500_baseline`: YOLOv8n-seg trained on Crack500
2. `v11_crack500_baseline`: YOLOv11n-seg trained on Crack500
3. `v8_deepcrack_baseline`: YOLOv8n-seg trained on DeepCrack
4. `v11_deepcrack_baseline`: YOLOv11n-seg trained on DeepCrack


In [ ]:
!mkdir -p configs utils distillation scripts checkpoints data/datasets data/teacher_logits_box data/teacher_logits_centroid runs


In [ ]:
!pip install -q ultralytics albumentations pycocotools thop pyyaml pandas


In [ ]:
%%writefile scripts/convert_crack500.py
#!/usr/bin/env python3
"""
Crack500 → YOLO seg format converter
=====================================
Crack500 structure:
  crack500/
  ├── traincrop/   ← 00001.jpg + 00001.png (binary mask, same stem)
  ├── valcrop/
  ├── testcrop/
  ├── train.txt    ← list of image filenames (optional)
  ├── val.txt
  └── test.txt

Output (YOLO seg format, ready for ultralytics):
  crack500_yolo/
  ├── images/
  │   ├── train/
  │   ├── val/
  │   └── test/
  ├── labels/
  │   ├── train/
  │   ├── val/
  │   └── test/
  └── dataset.yaml

Each .txt label: one line per connected crack instance
  0 x1 y1 x2 y2 ... (normalized polygon, class 0 = crack)

Usage:
  python scripts/convert_crack500.py \
      --src ~/distill/data/datasets/crack500 \
      --dst ~/distill/data/datasets/crack500_yolo
"""

import os
import cv2
import numpy as np
import argparse
import shutil
from pathlib import Path
from tqdm import tqdm


CLASS_ID = 0        # single class: crack
MIN_AREA = 50       # minimum pixel area to keep an instance
MIN_POINTS = 6      # minimum polygon points (3 coordinate pairs)


def binary_mask_to_yolo_instances(mask_path: str, img_w: int, img_h: int) -> list[str]:
    """
    Read binary PNG mask → split into instances via connectedComponents
    → convert each to normalized YOLO seg polygon string.

    Returns list of label lines (one per instance).
    """
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return []

    # Threshold (Crack500 masks are 0/255)
    binary = (mask > 127).astype(np.uint8)

    # Separate touching cracks into individual instances
    num_labels, labels_map = cv2.connectedComponents(binary)

    label_lines = []
    for label_id in range(1, num_labels):      # 0 = background
        instance = (labels_map == label_id).astype(np.uint8)

        if instance.sum() < MIN_AREA:
            continue

        # Find contours for this instance
        contours, _ = cv2.findContours(
            instance, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )

        for contour in contours:
            if len(contour) < MIN_POINTS // 2:
                continue

            # Flatten and normalize to [0, 1]
            pts = contour.squeeze()
            if pts.ndim == 1:
                pts = pts.reshape(1, 2)

            # Simplify contour slightly to reduce file size
            epsilon = 0.002 * cv2.arcLength(contour, True)
            simplified = cv2.approxPolyDP(contour, epsilon, True).squeeze()
            if simplified.ndim == 1:
                simplified = simplified.reshape(1, 2)
            if len(simplified) < 3:
                simplified = pts

            norm = []
            for x, y in simplified:
                norm.append(x / img_w)
                norm.append(y / img_h)

            if len(norm) < MIN_POINTS:
                continue

            coords_str = " ".join(f"{v:.6f}" for v in norm)
            label_lines.append(f"{CLASS_ID} {coords_str}")

    return label_lines


def process_split(src_dir: Path, dst_img_dir: Path, dst_lbl_dir: Path, split_name: str):
    """Process one split (train/val/test)."""

    # Crack500 stores images+masks together in traincrop/valcrop/testcrop
    crop_dir = src_dir / f"{split_name}crop"
    if not crop_dir.exists():
        # Try alternate names
        for candidate in [src_dir / split_name, src_dir / f"{split_name}data"]:
            if candidate.exists():
                crop_dir = candidate
                break
        else:
            print(f"  [WARNING] Could not find directory for split '{split_name}', skipping.")
            return 0

    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)

    # Find all images (jpg/jpeg/png that are NOT masks)
    all_files = sorted(crop_dir.iterdir())
    # Crack500: image = .jpg, mask = same stem + .png
    image_files = [f for f in all_files if f.suffix.lower() in ('.jpg', '.jpeg')
                   and ':Zone.Identifier' not in f.name]

    if not image_files:
        # Some versions store as .png images too — distinguish by paired files
        png_files = [f for f in all_files if f.suffix.lower() == '.png'
                     and ':Zone.Identifier' not in f.name]
        # If .jpg exists for a stem → .png is mask. If no .jpg → .png is image.
        jpg_stems = {f.stem for f in all_files if f.suffix.lower() in ('.jpg', '.jpeg')}
        image_files = [f for f in png_files if f.stem not in jpg_stems]

    converted = 0
    skipped   = 0

    for img_path in tqdm(image_files, desc=f"  {split_name}", leave=False):
        stem = img_path.stem

        # Find corresponding mask (.png with same stem)
        mask_path = crop_dir / f"{stem}.png"
        if not mask_path.exists():
            # Try .bmp
            mask_path = crop_dir / f"{stem}.bmp"
        if not mask_path.exists():
            skipped += 1
            continue

        # Read image to get dimensions
        img = cv2.imread(str(img_path))
        if img is None:
            skipped += 1
            continue
        h, w = img.shape[:2]

        # Convert mask to YOLO seg labels
        label_lines = binary_mask_to_yolo_instances(str(mask_path), w, h)

        # Copy image
        dst_img_path = dst_img_dir / img_path.name
        shutil.copy2(img_path, dst_img_path)

        # Write label file (even if empty — YOLO needs it)
        dst_lbl_path = dst_lbl_dir / f"{stem}.txt"
        with open(dst_lbl_path, "w") as f:
            f.write("\n".join(label_lines))

        converted += 1

    print(f"  {split_name}: {converted} images converted, {skipped} skipped")
    return converted


def write_dataset_yaml(dst: Path, num_train: int, num_val: int, num_test: int):
    """Write ultralytics-compatible dataset.yaml."""
    yaml_content = f"""# Crack500 — YOLO seg format
# Auto-generated by convert_crack500.py

path: {dst.resolve()}
train: images/train
val:   images/val
test:  images/test

nc: 1
names:
  0: crack

# Stats
# train: ~{num_train} images
# val:   ~{num_val} images
# test:  ~{num_test} images
"""
    with open(dst / "dataset.yaml", "w") as f:
        f.write(yaml_content)
    print(f"\n  dataset.yaml written to {dst / 'dataset.yaml'}")


def verify_conversion(dst: Path):
    """Quick sanity check on converted dataset."""
    print("\n[Verify] Checking converted dataset...")
    issues = 0
    for split in ["train", "val", "test"]:
        img_dir = dst / "images" / split
        lbl_dir = dst / "labels" / split
        if not img_dir.exists():
            continue

        imgs = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png"))
        lbls = list(lbl_dir.glob("*.txt"))

        # Check counts match
        if len(imgs) != len(lbls):
            print(f"  [!] {split}: {len(imgs)} images vs {len(lbls)} labels — mismatch!")
            issues += 1
        else:
            print(f"  {split}: {len(imgs)} images ✓")

        # Check a few labels are non-empty
        non_empty = sum(1 for l in lbls if l.stat().st_size > 0)
        empty     = len(lbls) - non_empty
        print(f"    labels with cracks: {non_empty} | empty (no crack): {empty}")

        if non_empty == 0:
            print(f"  [!] {split}: ALL labels are empty — check mask paths!")
            issues += 1

    if issues == 0:
        print("\n  ✓ Dataset looks good!")
    else:
        print(f"\n  ✗ {issues} issue(s) found — check output above.")

    return issues == 0


def main():
    parser = argparse.ArgumentParser(description="Convert Crack500 to YOLO seg format")
    parser.add_argument(
        "--src",
        type=str,
        required=True,
        help="Path to crack500 root dir (contains traincrop/, valcrop/, testcrop/)"
    )
    parser.add_argument(
        "--dst",
        type=str,
        default=None,
        help="Output directory (default: <src>_yolo)"
    )
    parser.add_argument(
        "--verify",
        action="store_true",
        default=True,
        help="Run sanity check after conversion"
    )
    args = parser.parse_args()

    src = Path(args.src).expanduser().resolve()
    dst = Path(args.dst).expanduser().resolve() if args.dst else src.parent / f"{src.name}_yolo"

    print(f"[Convert] Source: {src}")
    print(f"[Convert] Output: {dst}")
    print()

    if not src.exists():
        print(f"ERROR: Source directory not found: {src}")
        return

    counts = {}
    for split in ["train", "val", "test"]:
        n = process_split(
            src_dir    = src,
            dst_img_dir= dst / "images" / split,
            dst_lbl_dir= dst / "labels" / split,
            split_name = split,
        )
        counts[split] = n

    write_dataset_yaml(dst, counts["train"], counts["val"], counts["test"])

    if args.verify:
        verify_conversion(dst)

    print(f"\n[Done] Converted dataset at: {dst}")
    print(f"\nNext step — test YOLO11 loads it:")
    print(f"  from ultralytics import YOLO")
    print(f"  model = YOLO('yolo11n-seg.pt')")
    print(f"  model.train(data='{dst}/dataset.yaml', epochs=1, imgsz=512)")


if __name__ == "__main__":
    main()
# (appended — nothing, file is complete)


In [ ]:
%%writefile scripts/convert_deepcrack.py
#!/usr/bin/env python3
"""
DeepCrack → YOLO seg format converter
=====================================
DeepCrack structure:
  deepcrack/
  ├── train_img/      ← 11111.jpg
  ├── train_lab/      ← 11111.png (binary mask, 0/255)
  ├── test_img/       ← 111212-1.jpg
  └── test_lab/       ← 111212-1.png (binary mask, 0/255)

Output (YOLO seg format, ready for ultralytics):
  deepcrack_yolo/
  ├── images/
  │   ├── train/
  │   ├── val/
  │   └── test/
  ├── labels/
  │   ├── train/
  │   ├── val/
  │   └── test/
  └── dataset.yaml

Each .txt label: one line per connected crack instance
  0 x1 y1 x2 y2 ... (normalized polygon, class 0 = crack)

Splits:
  - Train: 80% of train_img (480 images)
  - Val: 20% of train_img (120 images)
  - Test: 100% of test_img (474 images)
"""

import os
import cv2
import numpy as np
import argparse
import shutil
from pathlib import Path
from tqdm import tqdm


CLASS_ID = 0        # single class: crack
MIN_AREA = 50       # minimum pixel area to keep an instance
MIN_POINTS = 6      # minimum polygon points (3 coordinate pairs)


def binary_mask_to_yolo_instances(mask_path: str, img_w: int, img_h: int) -> list[str]:
    """
    Read binary PNG mask → split into instances via connectedComponents
    → convert each to normalized YOLO seg polygon string.
    """
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return []

    # Threshold (DeepCrack masks are 0/255)
    binary = (mask > 127).astype(np.uint8)

    # Separate touching cracks into individual instances
    num_labels, labels_map = cv2.connectedComponents(binary)

    label_lines = []
    for label_id in range(1, num_labels):      # 0 = background
        instance = (labels_map == label_id).astype(np.uint8)

        if instance.sum() < MIN_AREA:
            continue

        # Find contours for this instance
        contours, _ = cv2.findContours(
            instance, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )

        for contour in contours:
            if len(contour) < MIN_POINTS // 2:
                continue

            # Flatten and normalize to [0, 1]
            pts = contour.squeeze()
            if pts.ndim == 1:
                pts = pts.reshape(1, 2)

            # Simplify contour slightly to reduce file size
            epsilon = 0.002 * cv2.arcLength(contour, True)
            simplified = cv2.approxPolyDP(contour, epsilon, True).squeeze()
            if simplified.ndim == 1:
                simplified = simplified.reshape(1, 2)
            if len(simplified) < 3:
                simplified = pts

            norm = []
            for x, y in simplified:
                norm.append(x / img_w)
                norm.append(y / img_h)

            if len(norm) < MIN_POINTS:
                continue

            coords_str = " ".join(f"{v:.6f}" for v in norm)
            label_lines.append(f"{CLASS_ID} {coords_str}")

    return label_lines


def process_images(image_list: list[Path], mask_dir: Path, dst_img_dir: Path, dst_lbl_dir: Path, split_name: str):
    """Process a list of images for a specific split."""
    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)

    converted = 0
    skipped = 0

    for img_path in tqdm(image_list, desc=f"  {split_name}", leave=False):
        stem = img_path.stem

        # Find corresponding mask (.png with same stem in mask_dir)
        mask_path = mask_dir / f"{stem}.png"
        if not mask_path.exists():
            skipped += 1
            continue

        # Read image to get dimensions
        img = cv2.imread(str(img_path))
        if img is None:
            skipped += 1
            continue
        h, w = img.shape[:2]

        # Convert mask to YOLO seg labels
        label_lines = binary_mask_to_yolo_instances(str(mask_path), w, h)

        # Copy image
        dst_img_path = dst_img_dir / img_path.name
        shutil.copy2(img_path, dst_img_path)

        # Write label file (even if empty — YOLO needs it)
        dst_lbl_path = dst_lbl_dir / f"{stem}.txt"
        with open(dst_lbl_path, "w") as f:
            f.write("\n".join(label_lines))

        converted += 1

    print(f"  {split_name}: {converted} images converted, {skipped} skipped")
    return converted


def write_dataset_yaml(dst: Path, num_train: int, num_val: int, num_test: int):
    """Write ultralytics-compatible dataset.yaml."""
    yaml_content = f"""# DeepCrack — YOLO seg format
# Auto-generated by convert_deepcrack.py

path: {dst.resolve()}
train: images/train
val:   images/val
test:  images/test

nc: 1
names:
  0: crack

# Stats
# train: ~{num_train} images
# val:   ~{num_val} images
# test:  ~{num_test} images
"""
    with open(dst / "dataset.yaml", "w") as f:
        f.write(yaml_content)
    print(f"\n  dataset.yaml written to {dst / 'dataset.yaml'}")


def verify_conversion(dst: Path):
    """Quick sanity check on converted dataset."""
    print("\n[Verify] Checking converted dataset...")
    issues = 0
    for split in ["train", "val", "test"]:
        img_dir = dst / "images" / split
        lbl_dir = dst / "labels" / split
        if not img_dir.exists():
            continue

        imgs = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png"))
        lbls = list(lbl_dir.glob("*.txt"))

        # Check counts match
        if len(imgs) != len(lbls):
            print(f"  [!] {split}: {len(imgs)} images vs {len(lbls)} labels — mismatch!")
            issues += 1
        else:
            print(f"  {split}: {len(imgs)} images ✓")

        # Check a few labels are non-empty
        non_empty = sum(1 for l in lbls if l.stat().st_size > 0)
        empty     = len(lbls) - non_empty
        print(f"    labels with cracks: {non_empty} | empty (no crack): {empty}")

        if non_empty == 0:
            print(f"  [!] {split}: ALL labels are empty — check mask paths!")
            issues += 1

    if issues == 0:
        print("\n  ✓ Dataset looks good!")
    else:
        print(f"\n  ✗ {issues} issue(s) found — check output above.")

    return issues == 0


def main():
    parser = argparse.ArgumentParser(description="Convert DeepCrack to YOLO seg format")
    parser.add_argument(
        "--src",
        type=str,
        default="data/datasets/deepcrack",
        help="Path to deepcrack root dir"
    )
    parser.add_argument(
        "--dst",
        type=str,
        default=None,
        help="Output directory (default: <src>_yolo)"
    )
    parser.add_argument(
        "--verify",
        action="store_true",
        default=True,
        help="Run sanity check after conversion"
    )
    args = parser.parse_args()

    src = Path(args.src).expanduser().resolve()
    dst = Path(args.dst).expanduser().resolve() if args.dst else src.parent / f"{src.name}_yolo"

    print(f"[Convert] Source: {src}")
    print(f"[Convert] Output: {dst}")
    print()

    if not src.exists():
        print(f"ERROR: Source directory not found: {src}")
        return

    # Check and clean output directory if exists
    if dst.exists():
        print(f"[Warning] Output directory exists, clearing: {dst}")
        shutil.rmtree(dst)

    # 1. Process Train split and do 80-20 partition
    train_img_dir = src / "train_img"
    train_lab_dir = src / "train_lab"
    
    if not train_img_dir.exists() or not train_lab_dir.exists():
        print(f"ERROR: Train directories not found under {src}")
        return

    all_train_files = sorted([
        f for f in train_img_dir.iterdir()
        if f.suffix.lower() in ('.jpg', '.jpeg', '.png')
        and ':Zone.Identifier' not in f.name
    ])

    # Deterministic shuffle
    rng = np.random.RandomState(42)
    shuffled_indices = rng.permutation(len(all_train_files))
    
    n_train = int(len(all_train_files) * 0.8)
    
    train_files = [all_train_files[i] for i in shuffled_indices[:n_train]]
    val_files   = [all_train_files[i] for i in shuffled_indices[n_train:]]

    print(f"Total training images: {len(all_train_files)}")
    print(f"  -> Train split: {len(train_files)}")
    print(f"  -> Val split:   {len(val_files)}")

    # 2. Process Test split
    test_img_dir = src / "test_img"
    test_lab_dir = src / "test_lab"
    
    if not test_img_dir.exists() or not test_lab_dir.exists():
        print(f"ERROR: Test directories not found under {src}")
        return

    test_files = sorted([
        f for f in test_img_dir.iterdir()
        if f.suffix.lower() in ('.jpg', '.jpeg', '.png')
        and ':Zone.Identifier' not in f.name
    ])
    print(f"Total test images:     {len(test_files)}")

    # Convert splits
    counts = {}
    counts["train"] = process_images(
        image_list = train_files,
        mask_dir   = train_lab_dir,
        dst_img_dir= dst / "images" / "train",
        dst_lbl_dir= dst / "labels" / "train",
        split_name = "train",
    )
    
    counts["val"] = process_images(
        image_list = val_files,
        mask_dir   = train_lab_dir,
        dst_img_dir= dst / "images" / "val",
        dst_lbl_dir= dst / "labels" / "val",
        split_name = "val",
    )

    counts["test"] = process_images(
        image_list = test_files,
        mask_dir   = test_lab_dir,
        dst_img_dir= dst / "images" / "test",
        dst_lbl_dir= dst / "labels" / "test",
        split_name = "test",
    )

    write_dataset_yaml(dst, counts["train"], counts["val"], counts["test"])

    if args.verify:
        verify_conversion(dst)

    print(f"\n[Done] Converted DeepCrack dataset at: {dst}")


if __name__ == "__main__":
    main()


In [ ]:
import os
import shutil
from pathlib import Path

# Priority check for custom dataset input folder
input_dir = Path("/kaggle/input/distill_datasetforme")
if not input_dir.exists():
    input_dir = Path("/kaggle/input")

print(f"Scanning input directory: {input_dir}")
if input_dir.exists():
    try:
        print("Direct contents:", os.listdir(str(input_dir)))
    except Exception as e:
        print("Error listing input directory:", e)

datasets_dir = Path("data/datasets")
datasets_dir.mkdir(parents=True, exist_ok=True)
checkpoints_dir = Path("checkpoints")
checkpoints_dir.mkdir(parents=True, exist_ok=True)

# 1. Clean up any existing local dataset directories/symlinks to avoid read-only collisions
for folder in ["combined_yolo", "crack500_yolo", "crack500_uncropped_yolo", "deepcrack_yolo", "crack500", "deepcrack"]:
    p = datasets_dir / folder
    if os.path.lexists(p):
        if os.path.islink(p): os.unlink(p)
        else: shutil.rmtree(p)

# 2. Clean up teacher logits folders and link them to /tmp to redirect disk usage
for folder in ["teacher_logits_box", "teacher_logits_centroid", "teacher_features"]:
    p_local = Path("data") / folder
    p_tmp = Path("/tmp") / folder
    if os.path.lexists(p_local):
        if os.path.islink(p_local): os.unlink(p_local)
        else: shutil.rmtree(p_local)
    p_tmp.mkdir(parents=True, exist_ok=True)
    os.symlink(p_tmp, p_local)
    print(f"Redirected {p_local} -> {p_tmp}")

# 3. Locate and link SAM 2 weights
linked_sam = False
for root, dirs, files in os.walk(str(input_dir)):
    root_path = Path(root)
    if "sam2_hiera_large.pt" in files and not linked_sam:
        dest = checkpoints_dir / "sam2_hiera_large.pt"
        if os.path.lexists(dest): os.remove(dest)
        os.symlink(root_path / "sam2_hiera_large.pt", dest)
        print(f"Linked SAM 2 checkpoint: {root_path / 'sam2_hiera_large.pt'} -> {dest}")
        linked_sam = True

if not (checkpoints_dir / "sam2_hiera_large.pt").exists():
    print("SAM 2 checkpoint not found in inputs. Downloading...")
    !wget -q https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt -O checkpoints/sam2_hiera_large.pt

# 4. Link raw datasets for conversion
linked_crack = False
linked_deep = False
for root, dirs, files in os.walk(str(input_dir)):
    root_path = Path(root)
    if "traincrop" in dirs and not linked_crack:
        dest = datasets_dir / "crack500"
        os.symlink(root_path, dest)
        print(f"Linked raw Crack500: {root_path} -> {dest}")
        linked_crack = True
    if "train_img" in dirs and not linked_deep:
        dest = datasets_dir / "deepcrack"
        os.symlink(root_path, dest)
        print(f"Linked raw DeepCrack: {root_path} -> {dest}")
        linked_deep = True

if not (datasets_dir / "crack500").exists():
    print("WARNING: Raw Crack500 dataset not linked!")
if not (datasets_dir / "deepcrack").exists():
    print("WARNING: Raw DeepCrack dataset not linked!")

In [ ]:
# Convert raw datasets to YOLO format separately
!python scripts/convert_crack500.py --src data/datasets/crack500 --dst data/datasets/crack500_yolo
!python scripts/convert_deepcrack.py --src data/datasets/deepcrack --dst data/datasets/deepcrack_yolo


## 🚀 Train Baselines (YOLOv8 vs YOLOv11 on Crack500 & DeepCrack)


In [ ]:
from ultralytics import YOLO

epochs = 150
imgsz = 512

print("=== 1. Training YOLOv8n-seg on Crack500 ===")
model_v8_c500 = YOLO("yolov8n-seg.pt")
model_v8_c500.train(
    data="data/datasets/crack500_yolo/dataset.yaml",
    epochs=epochs,
    imgsz=imgsz,
    project="runs/baselines",
    name="v8_crack500_baseline",
    seed=42
)

print("\n=== 2. Training YOLOv11n-seg on Crack500 ===")
model_v11_c500 = YOLO("yolo11n-seg.pt")
model_v11_c500.train(
    data="data/datasets/crack500_yolo/dataset.yaml",
    epochs=epochs,
    imgsz=imgsz,
    project="runs/baselines",
    name="v11_crack500_baseline",
    seed=42
)

print("\n=== 3. Training YOLOv8n-seg on DeepCrack ===")
model_v8_dc = YOLO("yolov8n-seg.pt")
model_v8_dc.train(
    data="data/datasets/deepcrack_yolo/dataset.yaml",
    epochs=epochs,
    imgsz=imgsz,
    project="runs/baselines",
    name="v8_deepcrack_baseline",
    seed=42
)

print("\n=== 4. Training YOLOv11n-seg on DeepCrack ===")
model_v11_dc = YOLO("yolo11n-seg.pt")
model_v11_dc.train(
    data="data/datasets/deepcrack_yolo/dataset.yaml",
    epochs=epochs,
    imgsz=imgsz,
    project="runs/baselines",
    name="v11_deepcrack_baseline",
    seed=42
)


## 📊 Comparative Evaluation Table


In [ ]:
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

models = {
    "YOLOv8n-seg (Crack500)": ("runs/baselines/v8_crack500_baseline/weights/best.pt", "data/datasets/crack500_yolo/dataset.yaml"),
    "YOLOv11n-seg (Crack500)": ("runs/baselines/v11_crack500_baseline/weights/best.pt", "data/datasets/crack500_yolo/dataset.yaml"),
    "YOLOv8n-seg (DeepCrack)": ("runs/baselines/v8_deepcrack_baseline/weights/best.pt", "data/datasets/deepcrack_yolo/dataset.yaml"),
    "YOLOv11n-seg (DeepCrack)": ("runs/baselines/v11_deepcrack_baseline/weights/best.pt", "data/datasets/deepcrack_yolo/dataset.yaml"),
}

results = []
for name, (ckpt, data_yaml) in models.items():
    if Path(ckpt).exists() and Path(data_yaml).exists():
        m = YOLO(ckpt)
        metrics = m.val(data=data_yaml, split="val")
        results.append({
            "Model": name,
            "mAP50-box": metrics.box.map50,
            "mAP50-95-box": metrics.box.map,
            "mAP50-seg": metrics.seg.map50,
            "mAP50-95-seg": metrics.seg.map,
            "Params (M)": sum(p.numel() for p in m.model.parameters()) / 1e6
        })

df = pd.DataFrame(results)
print("\n=== BASELINE COMPARISON SUMMARY ===")
print(df.to_string(index=False))
